In [5]:
from CAT import AcousticAttentionBlockConfig, AcousticAttentionBlock, CAT, CATConfig
from losses import CATLosses, LossCfg, PNSCfg
import torch
from torch import nn
from tqdm import tqdm
from MLP import MLPConfig, NeuralNetwork

In [ ]:
class CATTrain(nn.Module):
    def __init__(self):
        super().__init__()
        AAC_cfg = AcousticAttentionBlockConfig(
            d_model = 256,
            heads_mel = 8,
            heads_raw = 8,
            dropout = 0.1
        )

        CATConfig = CATConfig(
            d_model = 256,
            dropout = 0.1,
            acoustic_cfg = AAC_cfg,
            pooling = "flatten",
            mlp_config = MLPConfig(
                input = 256,
                output = 256,
                neurons=[32, 32, 32],
                layers= 3
            )
        )

        classifier_cfg = MLPConfig(
            input = 256,
            output = 4,
            neurons=[32, 32, 32],
            layers= 3
        )

        pns_cfg = PNSCfg(
            max_dims_per_step = None,
            different_class_only = True,
            eps = 1e-8,
            reduce = "mean"
        )

        losses_cfg = LossCfg(
            w_cls = 1.0,
            w_recon = 1.0,
            w_causal = 1.0,
            label_smoothing = 0.0,
            pns = pns_cfg
        )

        self.cat = CAT(CATConfig)


        #self.recon = 
        self.head = NeuralNetwork(classifier_cfg)

        self.losses = CATLosses(losses_cfg, self.head)

    def forward(self, x_mel, x_raw, pe3d, recon=True):

        cat_out = self.cat(x_mel, x_raw, pe3d)
        head_out = self.head(cat_out)

        if recon:
            recon_out = self.recon(cat_out)
            return head_out, recon_out
        
        return head_out
    
    def compute_losses(self, head_out, recon_out):
        self.losses(head_out)

        

In [4]:
X = torch.randn(4, 128, 16, 256)  # ejemplo input [B,T,P,D] según tus specs
Y_real = torch.randint(0, 10, (4,))
 
feats_cat  = cat(X)

z = cat.pool_features(feats_cat)

recon_pred = recon(feats_cat) 
head_pred = recon(z) 


loss_dict = losses(head_pred, Y_real, recon_pred, X, z)

loss_dict["loss_total"].backward()

TypeError: CAT.forward() missing 2 required positional arguments: 'x_raw' and 'pe3d'